# BÀI TẬP: TITANIC
**Nguồn:** kaggle.com/c/titanic (891 dòng)


In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from scipy import stats

sns.set_style('whitegrid')

csv_path = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv'

df = pd.read_csv(csv_path)
print('Loaded from:', csv_path)
df.head()

Loaded from: https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


---
# PHẦN A — DATA PROFILING
## A.1. Data size, column names, data types

In [2]:
print('Kích thước dữ liệu:', df.shape)
print('\nTên các cột:')
print(df.columns.tolist())
print('\nKiểu dữ liệu:')
print(df.dtypes)

Kích thước dữ liệu: (891, 15)

Tên các cột:
['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town', 'alive', 'alone']

Kiểu dữ liệu:
survived         int64
pclass           int64
sex                str
age            float64
sibsp            int64
parch            int64
fare           float64
embarked           str
class              str
who                str
adult_male        bool
deck               str
embark_town        str
alive              str
alone             bool
dtype: object


## A.2. Missing values & Duplicate data

In [3]:
print('Giá trị thiếu theo cột:')
print(df.isna().sum())
print('\nTỷ lệ thiếu (%):')
print((df.isna().mean() * 100).round(2))
print('\nSố dòng trùng lặp:', df.duplicated().sum())

Giá trị thiếu theo cột:
survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64

Tỷ lệ thiếu (%):
survived        0.00
pclass          0.00
sex             0.00
age            19.87
sibsp           0.00
parch           0.00
fare            0.00
embarked        0.22
class           0.00
who             0.00
adult_male      0.00
deck           77.22
embark_town     0.22
alive           0.00
alone           0.00
dtype: float64

Số dòng trùng lặp: 107


## A.3. Invalid values

In [4]:
invalid_checks = {
    'survived ngoài {0, 1}': ~df['survived'].isin([0, 1]),
    'pclass ngoài {1, 2, 3}': ~df['pclass'].isin([1, 2, 3]),
    'age âm': df['age'].notna() & (df['age'] < 0),
    'fare âm': df['fare'].notna() & (df['fare'] < 0),
    'sibsp âm': df['sibsp'] < 0,
    'parch âm': df['parch'] < 0,
}

print('Số giá trị không hợp lệ:')
print(pd.Series({name: mask.sum() for name, mask in invalid_checks.items()}))

Số giá trị không hợp lệ:
survived ngoài {0, 1}     0
pclass ngoài {1, 2, 3}    0
age âm                    0
fare âm                   0
sibsp âm                  0
parch âm                  0
dtype: int64


## A.4. Create a new column
Tạo cột `family_size` = sibsp + parch + 1.

In [5]:
df['family_size'] = df['sibsp'] + df['parch'] + 1
df[['sibsp', 'parch', 'family_size']].head()

,sibsp,parch,family_size
0,1,0,2
1,1,0,2
2,0,0,1
3,1,0,2
4,0,0,1


---
# PHẦN B — DESCRIPTIVE STATISTICS
## Group 1 — Central Tendency

In [6]:
central_tendency = pd.DataFrame({
    'mean': df[['age', 'fare']].mean(),
    'median': df[['age', 'fare']].median(),
    'mode': df[['age', 'fare']].mode().iloc[0]
})
central_tendency

,mean,median,mode
age,29.699118,28.0000,24.00
fare,32.204208,14.4542,8.05


## Group 2 — Dispersion

In [7]:
dispersion = pd.DataFrame({
    'std': df[['age', 'fare']].std(),
    'variance': df[['age', 'fare']].var(),
    'range': df[['age', 'fare']].max() - df[['age', 'fare']].min(),
    'IQR': df[['age', 'fare']].quantile(0.75) - df[['age', 'fare']].quantile(0.25)
})
dispersion

,std,variance,range,IQR
age,14.526497,211.019125,79.5800,17.8750
fare,49.693429,2469.436846,512.3292,23.0896


## Group 3 — Location and Shape

In [ ]:
location_shape = pd.DataFrame({
    'q25': df[['age', 'fare']].quantile(0.25),
    'q50': df[['age', 'fare']].quantile(0.50),
    'q75': df[['age', 'fare']].quantile(0.75),
    'skewness': df[['age', 'fare']].skew()
})
location_shape

---
# PHẦN C — DEFINE THE QUESTION

## Câu hỏi 1: Hạng vé nào có tỷ lệ sống sót cao nhất, chênh lệch bao nhiêu so với hạng thấp nhất?

In [8]:
survival_by_class = df.groupby('pclass', as_index=True)['survived'].mean().sort_values(ascending=False)
class_gap = survival_by_class.max() - survival_by_class.min()
print('Tỷ lệ sống sót theo hạng vé:')
print((survival_by_class * 100).round(2).astype(str) + '%')
print(f'Chênh lệch cao nhất - thấp nhất: {class_gap:.3f} ({class_gap:.1%})')

Tỷ lệ sống sót theo hạng vé:
pclass
1    62.96%
2    47.28%
3    24.24%
Name: survived, dtype: str
Chênh lệch cao nhất - thấp nhất: 0.387 (38.7%)


## Câu hỏi 2: Giới tính hay hạng vé ảnh hưởng đến sống sót mạnh hơn?

In [9]:
survival_by_sex = df.groupby('sex')['survived'].mean().sort_values(ascending=False)
survival_by_class = df.groupby('pclass')['survived'].mean().sort_values(ascending=False)
comparison = pd.DataFrame({
    'tỷ lệ sống sót': pd.concat([survival_by_sex, survival_by_class])
})
sex_gap = survival_by_sex.max() - survival_by_sex.min()
class_gap = survival_by_class.max() - survival_by_class.min()
print((comparison * 100).round(2).astype(str) + '%')
print(f'Chênh lệch theo giới tính: {sex_gap:.1%}')
print(f'Chênh lệch theo hạng vé: {class_gap:.1%}')

       tỷ lệ sống sót
female          74.2%
male           18.89%
1              62.96%
2              47.28%
3              24.24%
Chênh lệch theo giới tính: 55.3%
Chênh lệch theo hạng vé: 38.7%


## Câu hỏi 3: Vé đắt hơn có thực sự sống sót cao hơn không?

In [10]:
fare_survival_corr = df[['fare', 'survived']].corr().loc['fare', 'survived']
df['fare_group'] = pd.qcut(df['fare'], q=4, duplicates='drop')
survival_by_fare_group = df.groupby('fare_group', observed=True)['survived'].mean()
print(f'Tương quan Fare - Survived: {fare_survival_corr:.3f}')
print('Tỷ lệ sống sót theo nhóm giá vé:')
print((survival_by_fare_group * 100).round(2).astype(str) + '%')

Tương quan Fare - Survived: 0.257
Tỷ lệ sống sót theo nhóm giá vé:
fare_group
(-0.001, 7.91]     19.73%
(7.91, 14.454]     30.36%
(14.454, 31.0]      45.5%
(31.0, 512.329]    58.11%
Name: survived, dtype: str


## Câu hỏi 4: Gia đình đông người có ảnh hưởng đến khả năng sống sót không?

In [11]:
df['family_group'] = pd.cut(
    df['family_size'],
    bins=[0, 1, 4, np.inf],
    labels=['Một mình', 'Gia đình vừa', 'Gia đình đông']
)
survival_by_family = df.groupby('family_group', observed=False)['survived'].agg(['mean', 'count'])
survival_by_family.rename(columns={'mean': 'tỷ lệ sống sót', 'count': 'số hành khách'})

,tỷ lệ sống sót,số hành khách
family_group,,
Một mình,0.303538,537
Gia đình vừa,0.578767,292
Gia đình đông,0.161290,62


## Câu hỏi 5: Cảng lên tàu (embark_town) nào có tỷ lệ sống sót cao nhất?

In [12]:
survival_by_embark = df.groupby('embark_town', dropna=False)['survived'].agg(['mean', 'count']).sort_values('mean', ascending=False)
survival_by_embark.rename(columns={'mean': 'tỷ lệ sống sót', 'count': 'số hành khách'})

,tỷ lệ sống sót,số hành khách
embark_town,,
NaN,1.000000,2
Cherbourg,0.553571,168
Queenstown,0.389610,77
Southampton,0.336957,644


## Câu hỏi 6 (Tổng hợp) — Viết insight tổng hợp
Dựa trên Phần A, B, C, viết 4-5 câu insight tổng thể về dữ liệu Titanic.

## Insight tổng hợp
Dữ liệu có 891 hành khách; tỷ lệ sống sót chung khoảng 38%, cho thấy phần lớn hành khách không sống sót. Giới tính là yếu tố phân tách rõ nhất: tỷ lệ sống sót của nữ cao hơn nhiều so với nam, trong khi hạng vé cũng tạo ra chênh lệch đáng kể với lợi thế thuộc về hạng 1. Giá vé có mối tương quan dương với khả năng sống sót, và các nhóm trả vé cao hơn nhìn chung có tỷ lệ sống sót tốt hơn, dù đây chỉ là mối liên hệ chứ chưa chứng minh quan hệ nhân quả. Quy mô gia đình có xu hướng phi tuyến: đi cùng một gia đình nhỏ hoặc vừa thường tốt hơn đi một mình, còn gia đình quá đông có tỷ lệ sống sót thấp hơn. Trong các cảng lên tàu, Cherbourg có tỷ lệ sống sót cao nhất; kết quả này có thể phản ánh sự khác biệt về cơ cấu giới tính và hạng vé giữa các cảng.